# Corpus — parole utilisateur en voix variées (`fr_asr_user`)

**Runtime : GPU L4.**

Fait parler les tours *utilisateur* des dialogues existants par trois voix Voxtral différentes (l'oreille du modèle a besoin de diversité, contrairement à sa voix). Chaque clip est contre-transcrit et filtré, puis poussé vers `B_user_synth` sur le Hub. Reprend tout seul.


In [ ]:
# Jeton HF (écriture). Le plus propre : Colab > icône clé > secret HF_TOKEN.
import os
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("token lu depuis les secrets Colab")
except Exception:
    from getpass import getpass
    os.environ["HF_TOKEN"] = getpass("HF_TOKEN (write) : ")


In [ ]:
# Trois voix, trois tiers des tours user, en séquence.
import os, subprocess, urllib.request
url = "https://raw.githubusercontent.com/rcarvalo/finetuning_s2s_toolcalling/rd/pr_rca_eval_baseline/infra/colab_entrypoint.sh"
urllib.request.urlretrieve(url, "/content/entry.sh")
base = {
    "LFM2_BRANCH": "rd/pr_rca_eval_baseline", "LFM2_JOB": "build_brick_a",
    "BRICK_A_ENGINE": "voxtral", "BRICK_A_ROLE": "user",
    "BRICK_A_HF_PATH": "B_user_synth",
    "BRICK_A_BATCH": "32", "BRICK_A_CONCURRENCY": "16", "BRICK_A_PUSH_EVERY": "5",
    "LFM2_ROOT": "/content/repo", "LFM2_OUT": "/content/out",
}
for shard, voice in enumerate(["fr_male", "neutral_female", "casual_male"]):
    env = {**os.environ, **base, "BRICK_A_VOICE": voice, "BRICK_A_SHARD": f"{shard}/3"}
    print(f"=== {voice} (shard {shard}/3)", flush=True)
    subprocess.run(["bash", "/content/entry.sh"], env=env, check=False)
